In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Transpilare con i pass manager

<Accordion>
<AccordionItem title="Versioni dei pacchetti">

Il codice in questa pagina è stato sviluppato usando i seguenti requisiti.
Si raccomanda di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Il modo consigliato per transpilare un circuito è creare un staged pass manager e poi eseguire il suo metodo `run` con il circuito come input. Questa pagina spiega come transpilare circuiti quantistici in questo modo.
## Cos'è un (staged) pass manager?
Nel contesto di Qiskit SDK, la transpilazione si riferisce al processo di trasformazione di un circuito in input in una forma adatta all'esecuzione su un dispositivo quantistico. La transpilazione avviene tipicamente in una sequenza di passaggi chiamati transpiler pass. Il circuito viene elaborato da ogni transpiler pass in sequenza, con l'output di un pass che diventa l'input del successivo. Per esempio, un pass potrebbe scorrere il circuito e unire tutte le sequenze consecutive di gate a singolo qubit, e il pass successivo potrebbe sintetizzare questi gate nel set di base del dispositivo target. I transpiler pass inclusi in Qiskit si trovano nel modulo [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes).

Un pass manager è un oggetto che memorizza un elenco di transpiler pass e può eseguirli su un circuito. Per creare un pass manager, inizializza un [`PassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) con un elenco di transpiler pass. Per eseguire la transpilazione su un circuito, chiama il metodo [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) con un circuito come input.

Uno staged pass manager è un tipo speciale di pass manager che rappresenta un livello di astrazione superiore rispetto a un pass manager normale. Mentre un pass manager normale è composto da diversi transpiler pass, uno staged pass manager è composto da diversi *pass manager*. Si tratta di un'astrazione utile perché la transpilazione avviene tipicamente in fasi discrete, come descritto in [Stadi del transpiler](transpiler-stages), dove ogni fase è rappresentata da un pass manager. Gli staged pass manager sono rappresentati dalla classe [`StagedPassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager). Il resto di questa pagina descrive come creare e personalizzare (staged) pass manager.
## Generare uno staged pass manager preimpostato
Per creare uno staged pass manager preimpostato con impostazioni predefinite ragionevoli, usa la funzione [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager).

> **Note:** Il mock backend `FakeSherbrooke` di `qiskit_ibm_runtime` è usato in questi esempi, ma puoi provarlo su qualsiasi backend reale o fittizio compatibile con Qiskit. I tuoi risultati potrebbero essere diversi.

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

Per transpilare un circuito o un elenco di circuiti con un pass manager, passa il circuito o l'elenco di circuiti al metodo `run`. Proviamolo su un circuito a due qubit composto da un gate di Hadamard seguito da due gate CX adiacenti:

In [2]:
from qiskit import QuantumRegister, QuantumCircuit

# Create a circuit
qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)
a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

# Transpile it by calling the run method of the pass manager
transpiled = pass_manager.run(circuit)

# Draw it, excluding idle qubits from the diagram
transpiled.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/c1426e6c-f506-4938-8c0a-05198bae9746-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/c1426e6c-f506-4938-8c0a-05198bae9746-0.svg)

Consulta [Impostazioni predefinite e opzioni di configurazione della transpilazione](defaults-and-configuration-options) per una descrizione degli argomenti possibili della funzione `generate_preset_pass_manager`. Gli argomenti di `generate_preset_pass_manager` corrispondono agli argomenti della funzione [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<CodeAssistantAdmonition tagLine="Hai difficoltà a ricordare i dettagli del pass manager? Prova a chiedere a Qiskit Code Assistant." />

Se i pass manager preimpostati non soddisfano le tue esigenze, personalizza la transpilazione creando (staged) pass manager o persino transpiler pass personalizzati. Il resto di questa pagina descrive come creare pass manager. Per le istruzioni su come creare transpiler pass personalizzati, consulta [Scrivi il tuo transpiler pass](custom-transpiler-pass).

## Creare il tuo pass manager

Il modulo [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes) include molti transpiler pass che possono essere utilizzati per creare pass manager. Per creare un pass manager, inizializza un `PassManager` con un elenco di pass. Per esempio, il codice seguente crea un transpiler pass che unisce gate adiacenti a due qubit e poi li sintetizza in una base di gate [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate), [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate) e [$R_{xx}$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXXGate).

In [3]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (
    Collect2qBlocks,
    ConsolidateBlocks,
    UnitarySynthesis,
)

basis_gates = ["rx", "ry", "rxx"]
translate = PassManager(
    [
        Collect2qBlocks(),
        ConsolidateBlocks(basis_gates=basis_gates),
        UnitarySynthesis(basis_gates),
    ]
)

Per dimostrare questo pass manager in azione, testalo su un circuito a due qubit composto da un gate di Hadamard seguito da due gate CX adiacenti:

In [4]:
from qiskit import QuantumRegister, QuantumCircuit

qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)

a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

circuit.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/01317c54-68b5-4e41-893f-82ee223e22f0-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/01317c54-68b5-4e41-893f-82ee223e22f0-0.svg)

Per eseguire il pass manager sul circuito, chiama il metodo `run`.

In [5]:
translated = translate.run(circuit)
translated.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/019ad99b-bd38-4217-90ee-da43959dc8ad-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/019ad99b-bd38-4217-90ee-da43959dc8ad-0.svg)

Per un esempio più avanzato che mostra come creare un pass manager per implementare la tecnica di soppressione degli errori nota come dynamical decoupling, consulta [Creare un pass manager per il dynamical decoupling](dynamical-decoupling-pass-manager).

## Creare uno staged pass manager

Un `StagedPassManager` è un pass manager composto da singole fasi, dove ogni fase è definita da un'istanza di `PassManager`. Puoi creare un `StagedPassManager` specificando le fasi desiderate. Per esempio, il codice seguente crea uno staged pass manager con due fasi, `init` e `translation`. La fase `translation` è definita dal pass manager creato in precedenza.

In [6]:
from qiskit.transpiler import PassManager, StagedPassManager
from qiskit.transpiler.passes import UnitarySynthesis, Unroll3qOrMore

basis_gates = ["rx", "ry", "rxx"]
init = PassManager(
    [UnitarySynthesis(basis_gates, min_qubits=3), Unroll3qOrMore()]
)
staged_pm = StagedPassManager(
    stages=["init", "translation"], init=init, translation=translate
)

Non c'è limite al numero di fasi che puoi inserire in uno staged pass manager.

Un altro modo utile per creare uno staged pass manager è partire da uno staged pass manager preimpostato e poi sostituire alcune delle fasi. Per esempio, il codice seguente genera un pass manager preimpostato con livello di ottimizzazione 3 e poi specifica una fase `pre_layout` personalizzata.

In [7]:
import numpy as np
from qiskit.circuit.library import HGate, PhaseGate, RXGate, TdgGate, TGate
from qiskit.transpiler.passes import InverseCancellation

pass_manager = generate_preset_pass_manager(3, backend)
inverse_gate_list = [
    HGate(),
    (RXGate(np.pi / 4), RXGate(-np.pi / 4)),
    (PhaseGate(np.pi / 4), PhaseGate(-np.pi / 4)),
    (TGate(), TdgGate()),
]
logical_opt = PassManager(
    [
        InverseCancellation(inverse_gate_list),
    ]
)

# Add pre-layout stage to run extra logical optimization
pass_manager.pre_layout = logical_opt

Le [funzioni generatrici di stage](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#stage-generator-functions) possono essere utili per costruire pass manager personalizzati.
Generano fasi che forniscono funzionalità comuni usate in molti pass manager.
Per esempio, [`generate_embed_passmanager`](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#qiskit.transpiler.preset_passmanagers.generate_embed_passmanager) può essere usata per generare una fase
che "incorpora" un `Layout` iniziale selezionato da un layout pass nel dispositivo target specificato.

## Passi successivi
> **Tip:** - [Scrivi un transpiler pass personalizzato](custom-transpiler-pass).
> - [Crea un pass manager per il dynamical decoupling](dynamical-decoupling-pass-manager).
> - Per saperne di più sulla funzione `generate_preset_passmanager`, consulta l'argomento [Impostazioni predefinite e opzioni di configurazione della transpilazione](defaults-and-configuration-options).
> - Prova la guida [Confrontare le impostazioni del transpiler](/guides/circuit-transpilation-settings).
> - Consulta la [documentazione API del transpiler](https://docs.quantum.ibm.com/api/qiskit/transpiler).